# Lab - Model Choice: A Scorecard per Step

Three tasks that choose a model with numbers instead of habit: correctness, a typical time and the cost of a thousand requests, measured on the same cases - and a routing decision made step by step from those numbers.

| Task | What you learn |
| --- | --- |
| 1 | One case on two models: correct, seconds, dollars |
| 2 | The scorecard: twelve cases in parallel, median time, price per thousand |
| 3 | A second step, its own scorecard, and the cheapest model that meets the bar |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell runs offline; it answers from the policy in the prompt and, offline, takes a realistic time - the smaller model faster. In Task 3 it plays the smaller model as answering only the first part of a two-part question: a stand-in so the second scorecard has a difference to find, not a measurement of any real model.

**You do not write code from scratch.** Each cell already holds the code, with blanks marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import re
import statistics
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

load_dotenv()
client = Anthropic()
shopassist_lab.simulate_latency(True, per_token=0.004, first=0.25)   # offline only: a realistic pace

MODELS = ["claude-haiku-5-5", "claude-sonnet-5-5"]
# dollars per million tokens: input, output (pricing page)
PRICES = {"claude-haiku-5-5": (0.10, 0.50),
          "claude-sonnet-5-5": (2.00, 10.00)}
POLICY = Path("policy.txt").read_text()
SYSTEM = "You are ShopAssist. Answer from the policy in one sentence.\n\n" + POLICY

# question -> words a correct answer must contain
CASES = [
    ("Can I return earrings?", ["no"]),
    ("How long does a card refund take?", ["10 business days"]),
    ("Who approves a refund of 250?", ["human"]),
    ("Is the original shipping refunded for a wrong item?", ["yes"]),
    ("How many days do I have to report a damaged item?", ["7"]),
    ("Can I return opened cosmetics?", ["no"]),
    ("When is a lost return parcel treated as received?", ["14 days"]),
    ("Does a gift return refund the buyer's card?", ["no"]),
    ("Who reviews a warranty claim?", ["human"]),
    ("Can I return shoes without the shoe box?", ["no"]),
    ("I was in hospital and missed the 30 days. Can someone review my late return?", ["human"]),
    ("A sale item marked final sale: can I return it?", ["no"]),
]

---

## Task 1 - One case: correct, how long, what it cost

In [ ]:
# ============================================================
# TASK 1 - One case: correct, how long, what it cost
# ============================================================
#
# WHAT TO DO
#   Finish one_case(): start a timer, send the question, stop
#   the timer. The answer is correct when every required word
#   appears as a whole word. The cost comes from usage: input
#   tokens times the input price, plus output tokens times the
#   output price. Run one case on both models.
#
# WHY IT MATTERS
#   The default choice is usually the biggest model a team
#   trusts - and for many steps of an application that costs ten
#   or twenty times more than needed. A choice made with numbers
#   starts with three of them per case: correct, seconds,
#   dollars.
#
# WHERE TO SEE IT IN THE LECTURE
#   "And it works out the cost from the usage".
#
# HOW TO DO IT
#   Three blanks:
#
#       time.time() - start           how long the answer took
#       rf"\b{re.escape(w)}\b"         the word, as a whole word
#       usage.output_tokens * price_out    the output's share of the cost
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    def one_case(model, question, must):
        price_in, price_out = PRICES[model]
        start = time.time()
        response = client.messages.create(
            model=model, max_tokens=200, output_config={"effort": "low"}, system=SYSTEM,
            messages=[{"role": "user", "content": question}])
        seconds = ...   # time.time() - start
        answer = " ".join(b.text for b in response.content if b.type == "text").lower()
        ok = all(re.search(..., answer) for w in must)   # rf"\b{re.escape(w)}\b"
        usage = response.usage
        cost = (usage.input_tokens * price_in + ...) / 1e6   # usage.output_tokens * price_out
        return ok, seconds, cost

    first = {model: one_case(model, *CASES[1]) for model in MODELS}
    if any(... in result for result in first.values()):   # provided: said plainly
        raise TypeError("ellipsis")
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc) or "compiled pattern" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
for model, (ok, seconds, cost) in first.items():
    print(f"{model:18} correct {ok}  {seconds:.2f} s  ${cost:.6f}")

check("score_case", first=first)

---

## Task 2 - The scorecard: every case, both models

In [ ]:
# ============================================================
# TASK 2 - The scorecard: every case, both models
# ============================================================
#
# WHAT TO DO
#   Write run(): the twelve cases in parallel, six at a time,
#   and the scorecard row for one model - correct answers, the
#   median time, and the cost of a thousand requests. Print both
#   rows.
#
# WHY IT MATTERS
#   Same cases, same prompt, same check: the only difference is
#   the model. On these policy answers both are right every
#   time, and one costs twenty times more. That is a verdict
#   about this task with these cases - which is exactly the kind
#   of verdict a routing decision needs.
#
# WHERE TO SEE IT IN THE LECTURE
#   "That is twenty times more, for the same result on this
#   task".
#
# HOW TO DO IT
#   Three blanks:
#
#       sum(ok for ok, _, _ in results)          the correct answers
#       statistics.median(s for _, s, _ in results)   the typical time
#       cost / len(cases) * 1000                 the price of 1,000 requests
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    def run(model, cases):
        with ThreadPoolExecutor(max_workers=6) as pool:
            results = list(pool.map(lambda c: one_case(model, *c), cases))
        cost = sum(c for _, _, c in results)
        return {"correct": ..., "cases": len(cases),   # sum(ok for ok, _, _ in results)
                "p50_s": round(..., 2),   # statistics.median(s for _, s, _ in results)
                "per_1k": round(..., 2)}   # cost / len(cases) * 1000

    scorecard = {model: run(model, CASES) for model in MODELS}
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc) or "compiled pattern" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
except NameError as exc:   # provided: an earlier task has not run yet
    raise ValueError("{} - run the cells above first, Task 1 included."
                     .format(exc)) from None
print(f"{'model':18} {'correct':>8} {'p50 s':>6} {'$ per 1k requests':>18}")
for model, row in scorecard.items():
    print(f"{model:18} {row['correct']:>5}/{row['cases']} {row['p50_s']:6.2f} {row['per_1k']:18.2f}")

check("score_table", scorecard=scorecard)

---

## Task 3 - Each step gets the cheapest model that meets the bar

In [ ]:
# ============================================================
# TASK 3 - Each step gets the cheapest model that meets the bar
# ============================================================
#
# WHAT TO DO
#   Score a second step - questions that ask two things at once
#   - on both models. Then write route(): for each step, the
#   cheapest model whose correct share meets the bar.
#
# WHY IT MATTERS
#   The lecture's verdict is about one task. An application has
#   several, and each gets its own scorecard. Here the simulator
#   plays the smaller model as answering only the first part of
#   a two-part question - a stand-in, not a measurement - so the
#   second step needs the larger model. Routing is decided step
#   by step, from evidence, and re-run when cases or models
#   change.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Pick the cheapest model that meets the bar, step by step".
#
# HOW TO DO IT
#   Three blanks:
#
#       run(model, TWO_PART)                     the second step's scorecard
#       row["correct"] / row["cases"] >= BAR     meets the bar
#       min(passing, key=lambda m: rows[m]["per_1k"])   the cheapest of those
# ============================================================

BAR = 1.0   # provided: every case right - the business's bar for answers to customers
TWO_PART = [   # provided: question -> words the answer must contain
    ("How long does a card refund take, and who approves a refund of 250?",
     ["10 business days", "human"]),
    ("Can I return earrings, and how many days do I have to report a damaged item?", ["no", "7"]),
    ("When is a lost return parcel treated as received, and who reviews a warranty claim?",
     ["14 days", "human"]),
    ("Can I return opened cosmetics, and can I return shoes without the shoe box?",
     ["cosmetics", "shoe box"]),
]

try:
    # TODO: replace each ... below with the value named beside it
    steps = {"policy answers": scorecard,
             "two-part questions": {model: ... for model in MODELS}}   # run(model, TWO_PART)
    
    def route(rows):
        passing = [m for m, row in rows.items() if ...]   # row["correct"] / row["cases"] >= BAR
        return ... if passing else None   # min(passing, key=lambda m: rows[m]["per_1k"])
    
    routes = {step: route(rows) for step, rows in steps.items()}
    if ... in routes.values():   # provided: a blank left above, said plainly
        raise TypeError("ellipsis")
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc) or "compiled pattern" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
except NameError as exc:   # provided: an earlier task has not run yet
    raise ValueError("{} - run the cells above first, Tasks 1 to 2 included."
                     .format(exc)) from None
for step, rows in steps.items():
    for model, row in rows.items():
        print(f"{step:19} {model:18} {row['correct']}/{row['cases']}  ${row['per_1k']:.2f} per 1k")
    print(f"{step:19} -> {routes[step]}")

check("score_route", steps=steps, routes=routes)

---

## Done

Build a scorecard of correctness, latency and cost on your own cases, run every candidate model through it, and pick the cheapest model that meets the bar - step by step, and again whenever the cases or the models change.